# models

> The model catalog: curated ids per provider, and what live discovery adds.

This module owns the model catalog: a curated list per provider, in display order, and a live listing per provider that adds to it. It replaces three sources that each went stale on its own schedule: `fastllm.types.model_info_registry`, a hard-coded Claude tuple, and a generation-specific regex over OpenAI's live listing.

In [ ]:
#| default_exp models

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
from fastcore.test import test_eq, test_ne, test_fail

In [ ]:
#| export
import os, re
from fastcore.all import Path

## The catalog

`CATALOG` is what a picker offers before anything is asked over the network, in the order it is offered. `claude` is what Claude Code's `--model` accepts: the tier aliases that track the latest model, then the concrete ids. `anthropic` is the same models over the API. `codex` is what Codex's own picker lists.

In [ ]:
#| export
CATALOG = {
    'claude': ('fable', 'opus', 'sonnet', 'haiku', 'claude-fable-5-1', 'claude-opus-5-5', 'claude-sonnet-5', 'claude-sonnet-5-5', 'claude-haiku-4-5-20251001', 'claude-haiku-5-5'),
    'anthropic': ('claude-fable-5-1', 'claude-opus-5-5', 'claude-sonnet-5', 'claude-sonnet-5-5', 'claude-haiku-4-5-20251001', 'claude-haiku-5-5'),
    'openai': ('gpt-6-astra', 'gpt-6.1-sol', 'gpt-6-sol', 'gpt-6-luna', 'gpt-5.6', 'gpt-5.6-terra', 'gpt-5.6-sol', 'gpt-5.6-luna', 'gpt-5.4', 'gpt-5.4-mini'),
    'codex': ('gpt-6-astra', 'gpt-6-sol', 'gpt-6-luna', 'gpt-5.6-terra', 'gpt-5.6-sol', 'gpt-5.6-luna', 'gpt-5.5'),
    'gemini': ('gemini-3.5-flash', 'gemini-3.1-flash-lite'),
}

def claude_ids():
    "The concrete Claude Code ids in `CATALOG`, without the tier aliases."
    return tuple(m for m in CATALOG['claude'] if m.startswith('claude-'))

def claude_aliases():
    "Each Claude Code tier alias and the concrete id it tracks here."
    return {a: next(m for m in claude_ids() if m.startswith(f'claude-{a}-')) for a in CATALOG['claude'] if not a.startswith('claude-')}

In [ ]:
test_eq(claude_aliases()['fable'], 'claude-fable-5-1')
test_eq(claude_aliases()['opus'], 'claude-opus-5-5')
assert 'claude-haiku-5-5' in claude_ids() and 'claude-haiku-5-5' in CATALOG['anthropic']
test_eq(claude_aliases()['haiku'], 'claude-haiku-4-5-20251001')
assert 'claude-sonnet-5-5' in claude_ids() and claude_aliases()['sonnet'] == 'claude-sonnet-5'
assert 'gpt-6.1-sol' in CATALOG['openai'] and CATALOG['openai'][0] == 'gpt-6-astra'
test_eq(set(claude_aliases()), {'fable', 'opus', 'sonnet', 'haiku'})
for p, ids in CATALOG.items(): test_eq(len(ids), len(set(ids))), p

## Live discovery

Each provider that can be asked has one function. It returns what the provider lists, or `[]` when it cannot be asked: no key, no file, no network, a bad answer. OpenAI and Anthropic answer `/v1/models`. Codex is not asked over the network. Its picker list is the `models_cache.json` under `$CODEX_HOME` (default `~/.codex`), and the models marked `visibility: 'list'` are the ones it offers.

In [ ]:
#| export
def _listed(url, **headers):
    "The `id` of every row a `/v1/models` endpoint returns, or `[]`."
    try:
        import httpx2 as httpx
        r = httpx.get(url, headers=headers, timeout=10)
        r.raise_for_status()
        return [x.get('id', '') for x in r.json().get('data', [])]
    except Exception: return []

def openai_ids():
    "Every id the current OpenAI key can list, or `[]` when it cannot be asked."
    if not (key := os.getenv('OPENAI_API_KEY')): return []
    return _listed('https://api.openai.com/v1/models', Authorization=f'Bearer {key}')

def anthropic_ids():
    "Every id the current Anthropic key can list, or `[]` when it cannot be asked."
    if not (key := os.getenv('ANTHROPIC_API_KEY')): return []
    return _listed('https://api.anthropic.com/v1/models?limit=1000', **{'x-api-key': key, 'anthropic-version': '2023-06-01'})

def codex_ids():
    "What Codex's own picker lists, from its models cache, or `[]`."
    try: models = (Path(os.getenv('CODEX_HOME', '~/.codex')).expanduser()/'models_cache.json').read_json()['models']
    except Exception: return []
    return [m['slug'] for m in models if m.get('visibility') == 'list' and m.get('slug')]

LIVE = {'openai': openai_ids, 'anthropic': anthropic_ids, 'codex': codex_ids, 'claude': anthropic_ids}

In [ ]:
import json, os, tempfile
with tempfile.TemporaryDirectory() as d:
    cache = {'models': [{'slug': 'gpt-6-astra', 'visibility': 'list'}, {'slug': 'gpt-reserve', 'visibility': 'hide'}, {'slug': 'codex-auto-review', 'visibility': 'hide'}]}
    (Path(d)/'models_cache.json').write_text(json.dumps(cache))
    _old = os.environ.get('CODEX_HOME')
    os.environ['CODEX_HOME'] = d
    try: test_eq(codex_ids(), ['gpt-6-astra'])
    finally:
        if _old is None: del os.environ['CODEX_HOME']
        else: os.environ['CODEX_HOME'] = _old
    os.environ['CODEX_HOME'] = str(Path(d)/'nowhere')
    try: test_eq(codex_ids(), [])
    finally:
        if _old is None: del os.environ['CODEX_HOME']
        else: os.environ['CODEX_HOME'] = _old

## What a live listing may add

A provider lists everything it serves: dated snapshots, fine-tunes, realtime and audio variants, embeddings, moderation, image and search models. A picker wants none of those, whatever the generation. `_current` hides them by what they are rather than by which `gpt-N` they belong to, so a new generation appears the day it is listed. Older chat generations (`gpt-4*`, `gpt-3.5*`, `claude-3-*`) are opt-in through `legacy`.

In [ ]:
#| export
_HIDE = re.compile(r'-20\d\d-\d\d-\d\d|-\d{4}$|:|(?:^|-)(?:realtime|audio|image|transcribe|tts|live|instruct)(?:-|$)|embedding|moderation|dall-e|whisper|davinci|babbage|sora|computer-use|search-(?:api|preview)|deep-research')
_LEGACY = re.compile(r'^(?:gpt-4|gpt-3\.5|claude-[23]-)')

def _current(provider, mid, legacy=False):
    "Whether a listed id belongs in a picker; older generations only when `legacy`."
    return bool(re.search(r'\d', mid or '')) and not _HIDE.search(mid) and (legacy or not _LEGACY.match(mid))

In [ ]:
for ok in ('gpt-6-astra', 'gpt-7', 'o5-mini', 'claude-fable-5-1', 'claude-haiku-4-5-20251001', 'gemini-3.5-flash'): assert _current('openai', ok), ok
for hidden in ('gpt-6-astra-2026-09-01', 'gpt-3.5-turbo-0125', 'ft:gpt-3.5-turbo-0125:org::abc', 'ada:ft-org-2023-04-06-20-44-19', 'gpt-realtime', 'gpt-6-audio', 'gpt-image-2', 'gpt-6-transcribe', 'tts-1', 'gpt-live-1', 'gpt-3.5-turbo-instruct', 'text-embedding-4', 'omni-moderation-latest', 'dall-e-3', 'whisper-1', 'davinci-002', 'babbage-002', 'sora-2', 'computer-use-preview', 'gpt-5-search-api', 'gpt-4o-search-preview', 'o3-deep-research', 'chat-latest', ''):
    assert not _current('openai', hidden), hidden
    assert not _current('openai', hidden, legacy=True), hidden
for old in ('gpt-4.1', 'gpt-4o', 'gpt-3.5-turbo', 'claude-3-5-haiku-latest'):
    assert not _current('openai', old) and _current('openai', old, legacy=True), old

## What a picker offers

`provider_models` is the curated list first, then whatever the live listing adds that `_current` admits, sorted, with nothing repeated. `ramabana.core.probed` memoises the live answer for five minutes. A listing that fails or raises leaves the curated list.

In [ ]:
#| export
def provider_models(p, legacy=False, ttl=300):
    "Ids a picker offers for provider `p`: the curated ones in their order, then what live discovery adds."
    from ramabana.core import probed
    src = next((q for q, f in LIVE.items() if f is LIVE.get(p)), p)
    try: live = probed(f'{src}-models', lambda: LIVE[p](), ttl=ttl, disk=False) if p in LIVE else []
    except Exception: live = []
    newest = lambda m: (tuple(map(int, re.findall(r'\d+', m))), m)
    live = (m for m in live if _current(p, m, legacy) and not re.search(r'-20\d{6}$', m))
    return list(dict.fromkeys([*CATALOG.get(p, ()), *sorted(live, key=newest, reverse=True)]))

In [ ]:
from ramabana.core import forget_probes
_orig = dict(LIVE)
forget_probes()
LIVE['openai'] = lambda: ['gpt-6-astra', 'gpt-6-nova', 'gpt-6-nova-2026-09-01', 'gpt-6-audio', 'gpt-4.1']
try:
    got = provider_models('openai')
    test_eq(got[:len(CATALOG['openai'])], list(CATALOG['openai']))
    test_eq(got[len(CATALOG['openai']):], ['gpt-6-nova'])
    test_eq(got.count('gpt-6-astra'), 1)
    assert 'gpt-4.1' in provider_models('openai', legacy=True)
    # newest first, by the numbers: alphabetical put gpt-6.1 after every gpt-5
    forget_probes(); LIVE['openai'] = lambda: ['gpt-5.1', 'gpt-6.1-nova', 'gpt-5.10']
    test_eq(provider_models('openai')[len(CATALOG['openai']):], ['gpt-6.1-nova', 'gpt-5.10', 'gpt-5.1'])
finally: LIVE.clear(); LIVE.update(_orig); forget_probes()

In [ ]:
def _boom(): raise OSError('no network')
forget_probes()
LIVE.update({p: _boom for p in LIVE})
try:
    for p in CATALOG: test_eq(provider_models(p)[:len(CATALOG[p])], list(CATALOG[p]))
finally: LIVE.clear(); LIVE.update(_orig); forget_probes()
test_eq(provider_models('nobody'), [])

In [ ]:
_n, _key, _lst = [], os.environ.get('ANTHROPIC_API_KEY'), _listed
def _fake(*a, **kw): _n.append(1); return ['claude-sonnet-6', 'claude-sonnet-6-20270101', 'claude-sonnet-5-5']
forget_probes(); os.environ['ANTHROPIC_API_KEY'] = 'k'; _listed = _fake
try:
    got = provider_models('claude')
    test_eq(got[:len(CATALOG['claude'])], list(CATALOG['claude']))
    test_eq(got[len(CATALOG['claude']):], ['claude-sonnet-6'])
    assert 'claude-sonnet-6' in provider_models('anthropic') and len(_n) == 1
    forget_probes(); del os.environ['ANTHROPIC_API_KEY']
    test_eq(provider_models('claude'), list(CATALOG['claude']))
finally:
    _listed = _lst; forget_probes()
    if _key is None: os.environ.pop('ANTHROPIC_API_KEY', None)
    else: os.environ['ANTHROPIC_API_KEY'] = _key

## Through `available_models`

`ramabana.core.available_models` builds its hosted rows from `provider_models`, gated by `runtime_available` and `auth_status`. Each row has `value`, `label`, `provider` and `source`.

In [ ]:
import ramabana.models as _m
from ramabana.core import available_models, CLAUDE, CLAUDE_ALIASES, CLAUDE_MODELS
_orig_live, _key = dict(_m.LIVE), os.environ.get('OPENAI_API_KEY')
forget_probes()
_m.LIVE.update({'openai': lambda: ['gpt-6-astra', 'gpt-6-nova', 'gpt-6-nova-2026-09-01', 'gpt-6-audio'], 'anthropic': lambda: [], 'codex': lambda: []})
os.environ['OPENAI_API_KEY'] = 'sk-not-a-real-key'
try:
    rows = available_models()
    test_eq({tuple(sorted(r)) for r in rows}, {('label', 'provider', 'source', 'value')})
    openai = [r['label'] for r in rows if r['provider'] == 'openai']
    test_eq(openai, [*CATALOG['openai'], 'gpt-6-nova'])
    test_eq([r['value'] for r in rows if r['provider'] == 'openai'][0], 'openai/gpt-6-astra')
finally:
    _m.LIVE.clear(); _m.LIVE.update(_orig_live); forget_probes()
    if _key is None: del os.environ['OPENAI_API_KEY']
    else: os.environ['OPENAI_API_KEY'] = _key
test_eq(CLAUDE_MODELS, claude_ids())
assert {'claude-fable-5-1', 'claude-opus-5-5'} <= set(CLAUDE_MODELS)
test_eq(CLAUDE_ALIASES['fable'], 'claude-fable-5-1')
test_eq(CLAUDE['claude/opus'], 'claude-opus-5-5')